In [ ]:
# ============================================================
# PARTIE 1 — BASELINE : FAISS + SIMILARITY SEARCH
# ============================================================

import pickle
import pandas as pd

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


# ------------------------------------------------------------
# 1. Charger les données
# ------------------------------------------------------------

eval_df = pd.read_csv("../data/rag_evaluation_50_questions.csv")

with open("documents.pkl", "rb") as f:
    documents = pickle.load(f)

print("Nombre de documents :", len(documents))
print("Nombre de questions :", len(eval_df))


# ------------------------------------------------------------
# 2. Embedding model
# ------------------------------------------------------------

embedding_model = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-mpnet-base-v2",

    model_kwargs={
        "device": "cpu"
    },

    encode_kwargs={
        "normalize_embeddings": True
    }
)


# ------------------------------------------------------------
# 3. Construire FAISS
# ------------------------------------------------------------

vectorstore = FAISS.from_documents(

    documents=documents,

    embedding=embedding_model
)

print("Index FAISS créé.")


# ------------------------------------------------------------
# 4. Fonctions métriques
# ------------------------------------------------------------

def parse_relevant_ids(value):

    return set(
        str(value).split("|")
    )


def hit_at_k(
    retrieved_ids,
    relevant_ids
):

    return int(
        any(
            doc_id in relevant_ids
            for doc_id in retrieved_ids
        )
    )


def precision_at_k(
    retrieved_ids,
    relevant_ids
):

    relevant_count = sum(
        doc_id in relevant_ids
        for doc_id in retrieved_ids
    )

    return (
        relevant_count
        / len(retrieved_ids)
    )


def recall_at_k(
    retrieved_ids,
    relevant_ids
):

    relevant_count = sum(
        doc_id in relevant_ids
        for doc_id in retrieved_ids
    )

    return (
        relevant_count
        / len(relevant_ids)
    )


def reciprocal_rank(
    retrieved_ids,
    relevant_ids
):

    for rank, doc_id in enumerate(
        retrieved_ids,
        start=1
    ):

        if doc_id in relevant_ids:

            return 1 / rank

    return 0.0


# ------------------------------------------------------------
# 5. Fonction d'évaluation générique
# ------------------------------------------------------------

def evaluate_method(
    retrieve_function,
    eval_df,
    k=5
):

    results = []

    for _, row in eval_df.iterrows():

        question = row["question"]

        relevant_ids = parse_relevant_ids(
            row["relevant_doc_ids"]
        )

        docs = retrieve_function(
            question,
            k
        )

        retrieved_ids = [
            doc.metadata["doc_id"]
            for doc in docs
        ]

        results.append({

            "question_id":
                row["question_id"],

            "company":
                row["company"],

            "question":
                question,

            "question_type":
                row["question_type"],

            "expected_ids":
                row["relevant_doc_ids"],

            "retrieved_ids":
                "|".join(retrieved_ids),

            "hit":
                hit_at_k(
                    retrieved_ids,
                    relevant_ids
                ),

            "precision":
                precision_at_k(
                    retrieved_ids,
                    relevant_ids
                ),

            "recall":
                recall_at_k(
                    retrieved_ids,
                    relevant_ids
                ),

            "reciprocal_rank":
                reciprocal_rank(
                    retrieved_ids,
                    relevant_ids
                )
        })

    return pd.DataFrame(results)


# ------------------------------------------------------------
# 6. Résumé des résultats
# ------------------------------------------------------------

def summarize_results(results):

    return {

        "Hit@5":
            results["hit"].mean(),

        "Precision@5":
            results["precision"].mean(),

        "Recall@5":
            results["recall"].mean(),

        "MRR@5":
            results[
                "reciprocal_rank"
            ].mean()
    }


# ------------------------------------------------------------
# 7. Similarity Search
# ------------------------------------------------------------

def similarity_retrieve(
    query,
    k=5
):

    return vectorstore.similarity_search(
        query,
        k=k
    )


# ------------------------------------------------------------
# 8. Evaluation
# ------------------------------------------------------------

similarity_results = evaluate_method(

    retrieve_function=
        similarity_retrieve,

    eval_df=eval_df,

    k=5
)


similarity_summary = summarize_results(
    similarity_results
)


print("\nSIMILARITY RESULTS")

print(
    pd.DataFrame(
        [similarity_summary]
    )
)


# ------------------------------------------------------------
# 9. Questions ratées
# ------------------------------------------------------------

similarity_failed = similarity_results[
    similarity_results["hit"] == 0
]


print(
    "\nNombre de questions ratées :",
    len(similarity_failed)
)


display(
    similarity_failed[
        [
            "question_id",
            "company",
            "question",
            "expected_ids",
            "retrieved_ids"
        ]
    ]
)

Nombre de documents : 674
Nombre de questions : 50


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index FAISS créé.

SIMILARITY RESULTS
   Hit@5  Precision@5  Recall@5     MRR@5
0   0.84        0.172      0.82  0.649667

Nombre de questions ratées : 8


,question_id,company,question,expected_ids,retrieved_ids
3,Q04,Maybank,How did Maybank's net profit perform in 1QFY20...,1155.KL_805288_1,1155.KL_645403_2|1155.KL_691006_6|1155.KL_7129...
8,Q09,Maybank,How did Maybank's net profit change in 3QFY2025?,1155.KL_781707_1,1155.KL_691006_6|1155.KL_645403_2|1155.KL_7129...
19,Q20,Public Bank,What were Public Bank's 2QFY2026 net profit an...,1295.KL_815882_1,1295.KL_657034_3|1295.KL_556400_7|1295.KL_6343...
21,Q22,Public Bank,What net profit did Public Bank report for 1QF...,1295.KL_803671_1,1295.KL_692054_16|1295.KL_756153_1|1295.KL_567...
24,Q25,Public Bank,Why did Public Bank's 3QFY2025 net profit decl...,1295.KL_780949_1,1295.KL_449925_8|1295.KL_486752_1|1295.KL_4743...
38,Q39,Petronas Chemicals,What happened to PETRONAS Chemicals' shares in...,5183.KL_807113_4,5183.KL_815157_2|5183.KL_745434_1|5183.KL_8045...
47,Q48,Public Bank,Compare Public Bank's 2QFY2025 and 2QFY2026 ne...,1295.KL_768283_12|1295.KL_815882_1,1295.KL_794096_6|1295.KL_702805_4|1295.KL_6920...
48,Q49,Tenaga Nasional,Compare Tenaga Nasional's 2QFY2025 and 2QFY202...,5347.KL_768670_17|5347.KL_816073_1,5347.KL_657034_6|5347.KL_756961_1|5347.KL_7359...


In [26]:
# ============================================================
# PARTIE 2 — MMR
# Maximal Marginal Relevance
# ============================================================


# ------------------------------------------------------------
# 1. Fonction retrieval MMR
# ------------------------------------------------------------

def mmr_retrieve(
    query,
    k=5
):

    retriever = vectorstore.as_retriever(

        search_type="mmr",

        search_kwargs={

            # Nombre final de documents
            "k": k,

            # FAISS récupère d'abord
            # 20 candidats
            "fetch_k": 20,

            # 1 = priorité pertinence
            # 0 = priorité diversité
            "lambda_mult": 0.5
        }
    )

    return retriever.invoke(
        query
    )


# ------------------------------------------------------------
# 2. Evaluation MMR
# ------------------------------------------------------------

mmr_results = evaluate_method(

    retrieve_function=
        mmr_retrieve,

    eval_df=eval_df,

    k=5
)


mmr_summary = summarize_results(
    mmr_results
)


print("\nMMR RESULTS")

display(
    pd.DataFrame(
        [mmr_summary]
    )
)


# ------------------------------------------------------------
# 3. Questions ratées
# ------------------------------------------------------------

mmr_failed = mmr_results[
    mmr_results["hit"] == 0
]


print(
    "Nombre de questions ratées avec MMR :",
    len(mmr_failed)
)


display(
    mmr_failed[
        [
            "question_id",
            "company",
            "question",
            "expected_ids",
            "retrieved_ids"
        ]
    ]
)


# ------------------------------------------------------------
# 4. Résultats selon type de question
# ------------------------------------------------------------

print(
    "\nPerformance MMR par type de question"
)


display(

    mmr_results
    .groupby(
        "question_type"
    )[
        [
            "hit",
            "precision",
            "recall",
            "reciprocal_rank"
        ]
    ]
    .mean()
)


MMR RESULTS


,Hit@5,Precision@5,Recall@5,MRR@5
0,0.78,0.156,0.76,0.617333


Nombre de questions ratées avec MMR : 11


,question_id,company,question,expected_ids,retrieved_ids
9,Q10,CIMB,What was CIMB Group's 2QFY2026 net profit and ...,1023.KL_816234_6,1023.KL_609532_8|1023.KL_802036_9|1023.KL_8050...
14,Q15,CIMB,What drove CIMB Thai Bank's 8.4% rise in first...,1023.KL_800765_8,1023.KL_741917_3|1023.KL_669458_1|1023.KL_7748...
19,Q20,Public Bank,What were Public Bank's 2QFY2026 net profit an...,1295.KL_815882_1,1295.KL_657034_3|1155.KL_746018_1|1295.KL_7809...
21,Q22,Public Bank,What net profit did Public Bank report for 1QF...,1295.KL_803671_1,1295.KL_692054_16|1295.KL_474344_1|1295.KL_567...
27,Q28,Tenaga Nasional,What happened to Tenaga Nasional's 2QFY2026 ne...,5347.KL_816073_1,5347.KL_657034_6|5347.KL_735926_1|5347.KL_5016...
37,Q38,Petronas Chemicals,What were PETRONAS Chemicals' 2QFY2026 net pro...,5183.KL_815157_2,5183.KL_723169_9|5347.KL_756961_1|5183.KL_8045...
40,Q41,Petronas Chemicals,What loss did PETRONAS Chemicals report for 4Q...,5183.KL_793730_3,5183.KL_766472_1|5183.KL_752567_4|5183.KL_7817...
44,Q45,Petronas Chemicals,What were PETRONAS Chemicals' 4QFY2024 net pro...,5183.KL_745434_1,5183.KL_723169_9|1023.KL_794442_3|1295.KL_5564...
47,Q48,Public Bank,Compare Public Bank's 2QFY2025 and 2QFY2026 ne...,1295.KL_768283_12|1295.KL_815882_1,1295.KL_794096_6|1295.KL_567606_6|1295.KL_6920...
48,Q49,Tenaga Nasional,Compare Tenaga Nasional's 2QFY2025 and 2QFY202...,5347.KL_768670_17|5347.KL_816073_1,5347.KL_657034_6|5347.KL_735926_1|5347.KL_7463...



Performance MMR par type de question


,hit,precision,recall,reciprocal_rank
question_type,,,,
multi_doc_comparison,0.400000,0.080000,0.200000,0.400000
single_fact,0.822222,0.164444,0.822222,0.641481


In [ ]:
# ============================================================
# PARTIE 3 — FAISS + CROSS-ENCODER RERANKER
# ============================================================

import numpy as np

from sentence_transformers import CrossEncoder


# ------------------------------------------------------------
# 1. Charger le reranker
# ------------------------------------------------------------

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print(
    "Reranker chargé."
)


# ------------------------------------------------------------
# 2. Fonction FAISS + Reranking
# ------------------------------------------------------------

def rerank_retrieve(
    query,
    k=5
):


    candidates = (
        vectorstore.similarity_search(
            query,
            k=20
        )
    )

    pairs = [

        [
            query,
            doc.page_content
        ]

        for doc in candidates
    ]

    scores = reranker.predict(
        pairs
    )

    ranked_indices = np.argsort(
        scores
    )[::-1]

    reranked_documents = [

        candidates[i]

        for i in ranked_indices[:k]
    ]


    return reranked_documents


reranker_results = evaluate_method(

    retrieve_function=
        rerank_retrieve,

    eval_df=eval_df,

    k=5
)


reranker_summary = summarize_results(
    reranker_results
)


print(
    "\nRERANKER RESULTS"
)


display(
    pd.DataFrame(
        [reranker_summary]
    )
)


# ------------------------------------------------------------
# 4. Questions ratées
# ------------------------------------------------------------

reranker_failed = reranker_results[
    reranker_results["hit"] == 0
]


print(
    "Nombre de questions ratées avec reranker :",
    len(reranker_failed)
)


display(
    reranker_failed[
        [
            "question_id",
            "company",
            "question",
            "expected_ids",
            "retrieved_ids"
        ]
    ]
)

print(
    "\nPerformance Reranker par type de question"
)


display(

    reranker_results
    .groupby(
        "question_type"
    )[
        [
            "hit",
            "precision",
            "recall",
            "reciprocal_rank"
        ]
    ]
    .mean()
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker chargé.

RERANKER RESULTS


,Hit@5,Precision@5,Recall@5,MRR@5
0,0.88,0.176,0.84,0.799667


Nombre de questions ratées avec reranker : 6


,question_id,company,question,expected_ids,retrieved_ids
3,Q04,Maybank,How did Maybank's net profit perform in 1QFY20...,1155.KL_805288_1,1155.KL_645403_2|1155.KL_633896_1|1155.KL_7129...
8,Q09,Maybank,How did Maybank's net profit change in 3QFY2025?,1155.KL_781707_1,1155.KL_645403_2|1155.KL_691006_6|1155.KL_6338...
10,Q11,CIMB,Why did CIMB Group's 1QFY2026 net profit decli...,1023.KL_805091_5,1023.KL_513800_1|1023.KL_622242_7|1023.KL_4891...
21,Q22,Public Bank,What net profit did Public Bank report for 1QF...,1295.KL_803671_1,1295.KL_756153_1|1295.KL_567606_6|1295.KL_4049...
37,Q38,Petronas Chemicals,What were PETRONAS Chemicals' 2QFY2026 net pro...,5183.KL_815157_2,5183.KL_679750_3|5183.KL_723169_9|5183.KL_7454...
45,Q46,Maybank,Compare Maybank's profit performance in 4QFY20...,1155.KL_794278_1|1155.KL_805288_1,1155.KL_569719_7|1155.KL_702805_1|1155.KL_7567...



Performance Reranker par type de question


,hit,precision,recall,reciprocal_rank
question_type,,,,
multi_doc_comparison,0.800000,0.160000,0.400000,0.700000
single_fact,0.888889,0.177778,0.888889,0.810741


In [28]:
# ============================================================
# PARTIE 4 — COMPARAISON FINALE
# ============================================================


comparison_final = pd.DataFrame([

    {
        "Method":
            "FAISS Similarity",

        **similarity_summary,

        "Failed Questions":
            len(similarity_failed)
    },


    {
        "Method":
            "FAISS MMR",

        **mmr_summary,

        "Failed Questions":
            len(mmr_failed)
    },


    {
        "Method":
            "FAISS + Reranker",

        **reranker_summary,

        "Failed Questions":
            len(reranker_failed)
    }

])


display(
    comparison_final
)

,Method,Hit@5,Precision@5,Recall@5,MRR@5,Failed Questions
0,FAISS Similarity,0.84,0.172,0.82,0.649667,8
1,FAISS MMR,0.78,0.156,0.76,0.617333,11
2,FAISS + Reranker,0.88,0.176,0.84,0.799667,6
